# Ориентация текстовых кропов: воспроизводимое решение (версия 0.7538)

Ноутбук обучает компактную CNN для оценки вероятности поворота текста на 180° и создаёт `submission.csv` для 20 000 изображений. Для обучения используются TextOCR, для инференса — выданный архив, в котором нет целевых меток.

**Воспроизводимость.** Для близкого или точного повтора используйте те же файлы TextOCR, версии библиотек, параметры, GPU/CUDA и `test.zip`. CSV лучшей ранее отправленной версии сохранён в репозитории.

## Краткое описание подхода

1. Из TextOCR берутся читаемые горизонтальные word-box кропы: 150 000 из train и 25 000 из отдельного validation split. Выборка детерминирована seed 42.
2. Из каждого кропа создаётся копия, повёрнутая на 180°. Эти синтетические пары дают метки для обучения без ручной разметки тестовых данных.
3. С нуля обучается компактная CNN с depthwise-separable свёртками (97 729 параметров). Предобученных моделей, внешних API и LLM/VLM нет.
4. Температура логитов выбирается по Brier score на синтетических парах TextOCR validation. При инференсе сеть оценивает исходный кроп и его поворот; оценки симметризуются.
5. В CSV версии с публичным результатом `1 − Brier = 0.75382309` вероятность сжимается к 0.5: `p = 0.5 + 0.16957433201954025 × (p_raw − 0.5)`. Коэффициент подобран по агрегированной обратной связи публичного score предыдущей отправки, без доступа к индивидуальным тестовым меткам. Это раскрыто явно; TextOCR validation не является независимой оценкой на тесте.

**Данные и ПО:** TextOCR v0.1, PyTorch, NumPy, Pillow. Версии указаны в `outputs/requirements.txt`. Исходная разметка TextOCR не содержит поворотов на 180°; синтетические метки создаются вращением обучающих кропов.

## 1. Пути и параметры

Откройте ноутбук из корня репозитория `Avito-cv-test`. Если тестовый ZIP или TextOCR лежат в других местах, задайте переменные окружения `TEST_ZIP` и `TEXT_OCR_DIR`.

In [ ]:
from pathlib import Path
import csv, json, os, subprocess, sys, zipfile
import numpy as np
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'outputs' / 'solution_best075.py').exists():
    raise FileNotFoundError('Запустите ноутбук из корня клона Avito-cv-test')
TEXT_OCR_DIR = Path(os.environ.get('TEXT_OCR_DIR', '../textocr')).resolve()
TEST_ZIP = Path(os.environ.get('TEST_ZIP', '../test.zip')).resolve()
CACHE_DIR = TEXT_OCR_DIR / 'cache'
TRAIN_SCRIPT = PROJECT_ROOT / 'outputs' / 'solution_best075.py'
MODEL_PATH = PROJECT_ROOT / 'outputs' / 'best_075' / 'orientation_model.pt'
SUBMISSION_PATH = PROJECT_ROOT / 'outputs' / 'submission.csv'
REFERENCE_PATH = PROJECT_ROOT / 'outputs' / 'submission_best_known_07538.csv'
SEED, BATCH_SIZE, EPOCHS = 42, 256, 5
PROBABILITY_SHRINK = 0.16957433201954025
print('Python:', sys.version.split()[0])
print('Project:', PROJECT_ROOT)
print('TextOCR:', TEXT_OCR_DIR)
print('Test ZIP:', TEST_ZIP)

In [ ]:
required_paths = [TRAIN_SCRIPT, TEXT_OCR_DIR / 'TextOCR_0.1_train.json',
                  TEXT_OCR_DIR / 'TextOCR_0.1_val.json', TEXT_OCR_DIR / 'images', TEST_ZIP]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Не найдены входные данные:\n' + '\n'.join(missing))
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('CPU-обучение будет медленнее; для точного повтора используйте ту же CUDA/GPU-среду.')

## 2. Обучение

При первом запуске создаются кэши TextOCR, что может занять время. Затем модель обучается пять эпох и сохраняет веса с лучшим синтетическим validation Brier. При следующих запусках кэши переиспользуются. Seed фиксирован; детерминированный режим CUDA включён в скрипте.

In [ ]:
train_command = [sys.executable, str(TRAIN_SCRIPT), 'train',
                 '--textocr-dir', str(TEXT_OCR_DIR), '--cache-dir', str(CACHE_DIR),
                 '--model-out', str(MODEL_PATH), '--seed', str(SEED),
                 '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)]
print(' '.join(train_command))
subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)
metrics_path = MODEL_PATH.with_name('validation_metrics.json')
validation_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(validation_metrics, ensure_ascii=False, indent=2))

## 3. Предсказания и `submission.csv`

Для каждого `image_id` из `sample_submission.csv` PNG читается из ZIP. Модель оценивает исходное изображение и его поворот на 180°, вычисляет `p_raw`, применяет shrink-коэффициент и сохраняет строки в исходном порядке.

In [ ]:
predict_command = [sys.executable, str(TRAIN_SCRIPT), 'predict',
                   '--test-zip', str(TEST_ZIP), '--model', str(MODEL_PATH),
                   '--submission-out', str(SUBMISSION_PATH), '--batch-size', str(BATCH_SIZE),
                   '--probability-shrink', str(PROBABILITY_SHRINK)]
print(' '.join(predict_command))
subprocess.run(predict_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
# Проверяем схему, 20 000 строк, порядок ID и диапазон вероятностей.
with SUBMISSION_PATH.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['image_id', 'p_180']
    submission = list(reader)
assert len(submission) == 20_000
ids = [row['image_id'] for row in submission]
probabilities = [float(row['p_180']) for row in submission]
assert len(set(ids)) == len(ids)
assert all(np.isfinite(p) and 0 <= p <= 1 for p in probabilities)
with zipfile.ZipFile(TEST_ZIP) as zf:
    member = next(n for n in zf.namelist() if n.endswith('sample_submission.csv'))
    with zf.open(member) as sample_file:
        expected_ids = [row['image_id'] for row in csv.DictReader(
            line.decode('utf-8-sig') for line in sample_file)]
assert ids == expected_ids
print(f'Проверено: {len(submission):,} строк; колонки и диапазон вероятностей корректны.')
print('Файл:', SUBMISSION_PATH)
print(submission[:5])

## 4. Сверка с ранее отправленной версией

В репозитории сохранён CSV с публичным результатом `1 − Brier = 0.75382309`. Ячейка ниже показывает совпадение численных предсказаний и максимальное расхождение. Точный повтор может зависеть от версии PyTorch/CUDA и оборудования.

**Не вычисляйте Brier score на `test.zip`:** в архиве нет меток ориентации.

In [ ]:
if REFERENCE_PATH.exists():
    with REFERENCE_PATH.open(encoding='utf-8-sig', newline='') as f:
        reference = list(csv.DictReader(f))
    assert [r['image_id'] for r in reference] == ids
    delta = [abs(float(a['p_180']) - b) for a, b in zip(reference, probabilities)]
    print('Численные вероятности полностью совпадают:', all(d == 0 for d in delta))
    print('Максимальная разница p_180:', max(delta))
else:
    print('Референсный CSV отсутствует; проверьте результат по форматным проверкам выше.')

## Ограничения и интерпретация

- `validation_pair_brier` измерен на синтетических парах TextOCR, не на целевой тестовой разметке.
- `0.75382309` — публичный результат конкретного ранее загруженного CSV, а не локальная оценка по тестовым меткам.
- Shrink-коэффициент подбирался только по агрегированному публичному score. Индивидуальные тестовые метки и ручная разметка не использовались.
- Для независимой оценки нужны ориентированные кропы с метками, отложенные от обучения; ZIP задачи таких меток не содержит.